# Notebook 03 — Short-Read Alignment Concepts: BWT and BWA-MEM

**Module:** 09 — Genomics and NGS  
**Notebook:** 03 of 16  
**Time estimate:** 90 minutes

> Know the BWT conceptually. Know what MAPQ means precisely. Know the
> difference between BWA-MEM (DNA) and STAR/HISAT2 (RNA) — this is asked constantly.

---
## Step 1 — Motivation

Aligning 600 million Illumina reads against a 3 Gb reference genome must complete in
hours, not years. Smith-Waterman (O(n×m)) is too slow. BWA-MEM uses the
Burrows-Wheeler Transform to pre-index the reference, enabling O(q log G) query
per read rather than O(q × G).

---
## Step 2 — Intuition

**The core problem:** Given a 3 Gb reference and a 150 bp read, find all positions
in the reference where the read aligns within a small number of mismatches/gaps.

**BWT approach:** Transform the reference so that positions containing the same
characters cluster together. Then, given a query, binary-search through the clusters
to find matching positions. Like a sorted phone book — finding 'Smith' is O(log N),
not O(N).

**BWA-MEM algorithm:**
1. **SMEM (Super-Maximal Exact Match) seeding:** Find maximal exact matches between
   query and reference using the FM-index (BWT-based index).
2. **Chaining:** Connect seeds into chains covering the query.
3. **Gapped alignment:** Smith-Waterman around each chain to handle mismatches and gaps.
4. **MAPQ:** Score the mapping based on uniqueness.

---
## Step 3 — Biological Background

**Why spliced alignment is harder:**
RNA-seq reads come from mature mRNA — introns have been spliced out. A read from an
exon-exon junction does not align contiguously to the genome; it must be split across
a splice junction. BWA-MEM assumes contiguous alignment and will misalign or fail to
align junction reads. STAR and HISAT2 explicitly model splice junctions.

**STAR approach:**
1. Find maximal mappable prefix (MMP) of the read — the longest prefix that maps
   to the genome without gaps
2. If the MMP doesn't cover the full read, use the unmapped suffix to search for
   the donor splice site
3. Join the two alignment segments with the annotated intron

**HISAT2:** Graph-based — builds a graph genome including known splice junctions and
SNP variants, then aligns reads through the graph.

---
## Step 4 — Mathematical Explanation

**Burrows-Wheeler Transform (BWT):**

Given string $T$ of length $n$:
1. Form all rotations of $T\$$  (where $\$$ is a sentinel character < all others)
2. Sort the rotations lexicographically
3. BWT is the last column of the sorted rotation matrix

Property: the BWT of a real genome clusters repetitive characters together, enabling
fast search via the FM-index.

**FM-index backward search:** To find pattern $P$ in $T$:
1. Start with interval $[1, n]$ covering all BWT rows
2. For each character $c$ in $P$ (right to left):
   $$[lo, hi] = [C[c] + \text{Occ}(c, lo-1) + 1,\ C[c] + \text{Occ}(c, hi)]$$
   where $C[c]$ = number of characters in BWT smaller than $c$, and $\text{Occ}(c, i)$
   = number of occurrences of $c$ in BWT[1..i]
3. If $hi \geq lo$: $(hi - lo + 1)$ occurrences found

**MAPQ:** Phred-scaled probability that the alignment position is wrong:
$$\text{MAPQ} = -10 \log_{10}(P(\text{mapping is wrong}))$$

MAPQ=60 = probability wrong is $10^{-6}$ (very confident).  
MAPQ=0 = equally likely to map elsewhere (multi-mapper).  
MAPQ=255 = not available / not computed.

In [ ]:
# Step 6 — Python: BWT construction and FM-index
import numpy as np
from collections import defaultdict


def build_bwt(text: str, sentinel: str = '$') -> tuple[str, list[int]]:
    """
    Build the Burrows-Wheeler Transform of text.
    Returns (BWT string, suffix array).
    """
    t = text + sentinel
    n = len(t)
    # Build suffix array (naive O(n^2 log n) for illustration)
    suffixes = sorted(range(n), key=lambda i: t[i:])
    # BWT = character before each suffix in the sorted order
    bwt = ''.join(t[s - 1] if s > 0 else sentinel for s in suffixes)
    return bwt, suffixes


def build_fm_index(bwt: str) -> tuple[dict, dict]:
    """
    Build C (rank array) and Occ (occurrence matrix) for FM-index.
    C[c] = number of characters in BWT that are lexicographically smaller than c
    Occ[c][i] = number of occurrences of c in BWT[0..i]
    """
    chars = sorted(set(bwt))
    n = len(bwt)

    # C array
    counts = {c: bwt.count(c) for c in chars}
    C = {}
    cum = 0
    for c in chars:
        C[c] = cum
        cum += counts[c]

    # Occ matrix
    Occ = {c: [0] * (n + 1) for c in chars}
    for i, b in enumerate(bwt):
        for c in chars:
            Occ[c][i + 1] = Occ[c][i] + (1 if b == c else 0)

    return C, Occ


def fm_search(pattern: str, bwt: str, C: dict, Occ: dict) -> tuple[int, int]:
    """
    FM-index backward search. Returns interval [lo, hi) in SA.
    hi - lo = number of occurrences.
    """
    n = len(bwt)
    lo, hi = 0, n

    for c in reversed(pattern):
        if c not in C:
            return 0, 0  # character not in reference
        lo = C[c] + Occ[c][lo]
        hi = C[c] + Occ[c][hi]
        if lo >= hi:
            return 0, 0

    return lo, hi


def fm_find_positions(pattern: str, bwt: str, sa: list, C: dict, Occ: dict) -> list[int]:
    lo, hi = fm_search(pattern, bwt, C, Occ)
    return sorted(sa[i] for i in range(lo, hi))


# Test on a small reference
ref = 'ATCGATCGATCG'
bwt, sa = build_bwt(ref)
C, Occ = build_fm_index(bwt)

print(f"Reference: {ref}")
print(f"BWT:       {bwt}")
print(f"SA:        {sa}")
print()

for pattern in ['ATCG', 'CGA', 'TTTT', 'TCG']:
    positions = fm_find_positions(pattern, bwt, sa, C, Occ)
    # Verify with naive search
    naive = [i for i in range(len(ref)) if ref[i:i+len(pattern)] == pattern]
    print(f"Pattern '{pattern}': FM-index={positions}, naive={naive}, match={positions==naive}")

In [ ]:
# MAPQ: simulate and visualise
import matplotlib.pyplot as plt
import numpy as np


def mapq_from_scores(best_score: int, second_best_score: int, max_mapq: int = 60) -> int:
    """
    Simplified MAPQ: based on the gap between best and second-best alignment score.
    Higher gap → higher confidence → higher MAPQ.
    """
    if second_best_score is None:
        return max_mapq  # unique mapper
    score_diff = best_score - second_best_score
    if score_diff <= 0:
        return 0  # equally good alternative
    # Approximate formula used by BWA-MEM
    mapq = int(min(max_mapq, 6 * score_diff))
    return mapq


# MAPQ interpretation guide
mapq_scenarios = [
    (0, 'Multi-mapper: equal alignments elsewhere'),
    (1, 'Very likely multi-mapper'),
    (10, '90% confidence in primary alignment'),
    (20, '99% confidence (use for germline SNPs)'),
    (30, '99.9% confidence'),
    (60, 'Unique mapper: maximally confident (BWA-MEM max)'),
    (255, 'MAPQ not available (e.g., unmapped)'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: MAPQ → probability of wrong mapping
ax = axes[0]
mapq_vals = np.arange(0, 61)
p_wrong = 10 ** (-mapq_vals / 10)
ax.semilogy(mapq_vals, p_wrong, 'b-', lw=2)
ax.axhline(0.01, color='orange', ls='--', label='1% error (MAPQ 20)')
ax.axhline(0.001, color='green', ls='--', label='0.1% error (MAPQ 30)')
ax.set_xlabel('MAPQ score')
ax.set_ylabel('P(mapping is wrong)')
ax.set_title('A. MAPQ → mapping error probability\n(Phred-scaled)')
ax.legend(fontsize=8)

# Panel B: Distribution of MAPQ values in a typical WGS run
ax2 = axes[1]
rng = np.random.default_rng(42)
# Simulate realistic MAPQ distribution: most reads unique, some multi-mappers
mapqs = np.concatenate([
    rng.choice([0, 1, 3], size=50000, p=[0.7, 0.2, 0.1]),   # multi-mappers ~5%
    rng.integers(20, 61, size=950000)                          # unique mappers ~95%
])
ax2.hist(mapqs, bins=61, range=(0, 61), color='steelblue', edgecolor='none')
ax2.set_xlabel('MAPQ')
ax2.set_ylabel('Number of reads')
ax2.set_title('B. Simulated MAPQ distribution\n(typical 30× WGS: ~5% multi-mappers)')
ax2.axvline(20, color='orange', ls='--', label='Common filter threshold (MAPQ≥20)')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('alignment_concepts.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Multi-mappers (MAPQ<10): {(mapqs < 10).sum():,} ({(mapqs < 10).mean()*100:.1f}%)")
print(f"High-confidence (MAPQ≥20): {(mapqs >= 20).sum():,} ({(mapqs >= 20).mean()*100:.1f}%)")

---
## BWA-MEM vs STAR: the one-sentence answer

| Tool | Use for | Key difference |
|------|---------|----------------|
| BWA-MEM | Genomic DNA (WGS, WES, ChIP) | Contiguous alignment only; no splice junctions |
| STAR | RNA-seq | Discovers and uses splice junctions; two-pass mode for novel junctions |
| HISAT2 | RNA-seq | Graph-based; lower memory than STAR; includes SNP-aware alignment |
| Bowtie2 | Short DNA reads, ChIP-seq | Faster than BWA for very short reads (<50 bp); no gaps by default |
| minimap2 | Long reads (PacBio, ONT) | Handles high error rate and long gaps; also does spliced alignment |

---
## Step 8 — Exercises

1. Construct the BWT of 'BANANA' by hand. Verify with the implementation above.
2. Using the FM-index, find all occurrences of 'ANA' in 'BANANA'. Show the backward
   search steps.
3. A read has MAPQ=0. What does this mean for variant calling? Should you include this
   read in a pileup?
4. An RNA-seq read spans an exon-exon junction. Why would BWA-MEM fail to align it
   while STAR would succeed?

---
## Step 10 — Quiz

1. What is the BWT and why is it useful for sequence alignment?
2. What does MAPQ=60 mean? What does MAPQ=0 mean?
3. Why use STAR instead of BWA for RNA-seq?
4. What is a SMEM (Super-Maximal Exact Match) in BWA-MEM?
5. At what MAPQ threshold do most variant callers filter reads?

---
## Step 12 — Reflection

> *[Explain in one paragraph: why does pre-indexing a genome with the BWT make
> alignment so much faster than running Smith-Waterman at every position?]*

---
*Next: `04_sam_bam_format_deep_dive.ipynb`*